# Callinf kserve model for inference

import os
import json
import joblib
import requests
from minio import Minio
from kafka import KafkaConsumer
import numpy as np
import pandas as pd
from io import BytesIO

# Kafka settings
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "kafka.apache-kafka.svc.cluster.local:9092")
KAFKA_TOPIC = os.getenv("KAFKA_TOPIC", "sliding_window_lstm")
KAFKA_GROUP_ID = os.getenv("KAFKA_GROUP_ID", "lstm-cpu-infer")
INFERENCE_URL = os.getenv(
    "INFERENCE_URL",
    "http://inference-serving-kserve-cpu-pct.lstm-iqu.svc.cluster.local/v2/models/inference-serving-kserve-cpu-pct/infer"
)

# MinIO settings: match your deployment
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "minio-service.kubeflow.svc.cluster.local:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY", "minio")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY", "minio123")
MINIO_SECURE = os.getenv("MINIO_SECURE", "False").lower() == "true"
MINIO_BUCKET = os.getenv("MINIO_BUCKET", "k8s-resources-forecast")
MINIO_SCALER_OBJECT = os.getenv(
    "MINIO_SCALER_OBJECT",
    "data/k8s-preprocessed/node-1-cpu-scaler/node-1-scaler.pkl"
)
SCALER_FEATURE_NAME = os.getenv("SCALER_FEATURE_NAME", "ac3-node-1-vm_cpu_pct")  # Set to your actual column

def fetch_scaler_from_minio():
    """
    Download the scaler from MinIO and load it using joblib.
    """
    client = Minio(
        MINIO_ENDPOINT,
        access_key=MINIO_ACCESS_KEY,
        secret_key=MINIO_SECRET_KEY,
        secure=MINIO_SECURE,
    )
    print(f"Downloading scaler from bucket '{MINIO_BUCKET}' object '{MINIO_SCALER_OBJECT}' ...")
    data = client.get_object(MINIO_BUCKET, MINIO_SCALER_OBJECT)
    scaler = joblib.load(BytesIO(data.read()))
    data.close()
    print("Scaler loaded from MinIO.")
    return scaler

def scale_window(window, scaler, feature_name):
    """
    Convert list of dicts to DataFrame with column, then scale.
    """
    cpu_vals = np.array([d["cpu_pct"] for d in window]).reshape(-1, 1)
    cpu_df = pd.DataFrame(cpu_vals, columns=[feature_name])
    scaled_vals = scaler.transform(cpu_df)
    return scaled_vals.flatten().tolist()

def send_inference(window_scaled):
    """
    Send the scaled window to the inference server and return the prediction.
    """
    payload = {
        "inputs": [{
            "name": "input-0",
            "shape": [1, len(window_scaled), 1],
            "datatype": "FP32",
            "data": [[ [float(x)] for x in window_scaled ]]
        }]
    }
    try:
        resp = requests.post(INFERENCE_URL, json=payload, timeout=10)
        if resp.status_code == 200:
            try:
                result = resp.json()
                # Adjust as needed for your model output structure!
                pred = result["outputs"][0]["data"][0]
                return pred
            except Exception as e:
                print(f"[ERROR] Failed to parse inference response: {e}")
                print("Body:", resp.text)
                return None
        else:
            print(f"[ERROR] Inference server HTTP {resp.status_code}: {resp.text}")
            return None
    except Exception as ex:
        print(f"[ERROR] Exception in requests: {ex}")
        return None

def main():
    scaler = fetch_scaler_from_minio()

    consumer = KafkaConsumer(
        KAFKA_TOPIC,
        bootstrap_servers=[s.strip() for s in KAFKA_BOOTSTRAP_SERVERS.split(",") if s.strip()],
        group_id=None,
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        auto_offset_reset="latest",
        enable_auto_commit=False
    )

    print(f"Subscribed to topic '{KAFKA_TOPIC}'. Waiting for windows...")
    for msg in consumer:
        window = msg.value
        print("\nReceived window:", " | ".join(f"{d['ts'].split()[-1]} {d['cpu_pct']:.2f}" for d in window))
        try:
            window_scaled = scale_window(window, scaler, SCALER_FEATURE_NAME)
            print("\nScaled window:", " | ".join(f"{x:.4f}" for x in window_scaled))
            pred = send_inference(window_scaled)
            print("\nForecasted value (scaled):", pred)
            if pred is not None:
                # Inverse transform to original CPU % (always use a DataFrame to avoid warnings)
                pred_rescaled = scaler.inverse_transform(
                    pd.DataFrame([[pred]], columns=[SCALER_FEATURE_NAME])
                )[0, 0]
                print("\nForecasted value (rescaled):", pred_rescaled)
                print("-----------------------------------------------------------------------------------------------\n")
                print("\n")
        except Exception as e:
            print("[ERROR] During processing window:", e)

if __name__ == "__main__":
    main()


# V2 extend the metrics and kafka publishing fro them
n-memory buffer to store predictions until ground truth arrives

Second Kafka producer for metrics

Publishing to two topics:

lstm_forecast_values (for forecasted + real value)

lstm_forecast_metrics (for error + inference latency)

Full debug printing

You just need to provide the ground truth value when it arrives, for now this example simulates the ground truth as the last element of the window (as a placeholder). You can replace this logic as needed.

In [ ]:
import os
import json
import joblib
import requests
from minio import Minio
from kafka import KafkaConsumer, KafkaProducer
import numpy as np
import pandas as pd
from io import BytesIO
import time
from collections import deque

# --- Config ---
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "kafka.apache-kafka.svc.cluster.local:9092")
KAFKA_TOPIC = os.getenv("KAFKA_TOPIC", "sliding_window_lstm")
INFERENCE_URL = os.getenv(
    "INFERENCE_URL",
    "http://inference-serving-kserve-cpu-pct.lstm-iqu.svc.cluster.local/v2/models/inference-serving-kserve-cpu-pct/infer"
)
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "minio-service.kubeflow.svc.cluster.local:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY", "minio")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY", "minio123")
MINIO_SECURE = os.getenv("MINIO_SECURE", "False").lower() == "true"
MINIO_BUCKET = os.getenv("MINIO_BUCKET", "k8s-resources-forecast")
MINIO_SCALER_OBJECT = os.getenv(
    "MINIO_SCALER_OBJECT",
    "data/k8s-preprocessed/node-1-cpu-scaler/node-1-scaler.pkl"
)
SCALER_FEATURE_NAME = os.getenv("SCALER_FEATURE_NAME", "ac3-node-1-vm_cpu_pct")
FORECAST_TOPIC = os.getenv("FORECAST_TOPIC", "lstm_forecast_values")
METRICS_TOPIC = os.getenv("METRICS_TOPIC", "lstm_forecast_metrics")
FORECAST_HORIZON_SEC = 5   # <--- Adjust this if your horizon is NOT 5 seconds

def fetch_scaler_from_minio():
    client = Minio(
        MINIO_ENDPOINT,
        access_key=MINIO_ACCESS_KEY,
        secret_key=MINIO_SECRET_KEY,
        secure=MINIO_SECURE,
    )
    print(f"Downloading scaler from bucket '{MINIO_BUCKET}' object '{MINIO_SCALER_OBJECT}' ...")
    data = client.get_object(MINIO_BUCKET, MINIO_SCALER_OBJECT)
    scaler = joblib.load(BytesIO(data.read()))
    data.close()
    print("Scaler loaded from MinIO.")
    return scaler

def scale_window(window, scaler, feature_name):
    cpu_vals = np.array([d["cpu_pct"] for d in window]).reshape(-1, 1)
    cpu_df = pd.DataFrame(cpu_vals, columns=[feature_name])
    scaled_vals = scaler.transform(cpu_df)
    return scaled_vals.flatten().tolist()

def send_inference(window_scaled):
    payload = {
        "inputs": [{
            "name": "input-0",
            "shape": [1, len(window_scaled), 1],
            "datatype": "FP32",
            "data": [[ [float(x)] for x in window_scaled ]]
        }]
    }
    try:
        resp = requests.post(INFERENCE_URL, json=payload, timeout=10)
        if resp.status_code == 200:
            result = resp.json()
            return result["outputs"][0]["data"][0]
        else:
            print(f"[ERROR] Inference server HTTP {resp.status_code}: {resp.text}")
            return None
    except Exception as ex:
        print(f"[ERROR] Exception in requests: {ex}")
        return None

def publish_forecast_and_metric(producer, pred_for, pred_made_at, forecasted_value, real_value, error, latency):
    msg_forecast = {
        "prediction_for": pred_for,
        "prediction_made_at": pred_made_at,
        "forecasted_value": forecasted_value,
        "real_value": real_value
    }
    try:
        print("[DEBUG] Publishing forecast to topic:", FORECAST_TOPIC)
        print(msg_forecast)
        producer.send(FORECAST_TOPIC, msg_forecast)
        producer.flush()
    except Exception as e:
        print("[ERROR] Failed to publish forecast:", e)

    if error is not None and latency is not None:
        msg_metrics = {
            "prediction_for": pred_for,
            "error": error,
            "latency_ms": latency
        }
        try:
            print("[DEBUG] Publishing metrics to topic:", METRICS_TOPIC)
            print(msg_metrics)
            producer.send(METRICS_TOPIC, msg_metrics)
            producer.flush()
        except Exception as e:
            print("[ERROR] Failed to publish metrics:", e)

def main():
    scaler = fetch_scaler_from_minio()
    consumer = KafkaConsumer(
        KAFKA_TOPIC,
        bootstrap_servers=[s.strip() for s in KAFKA_BOOTSTRAP_SERVERS.split(",") if s.strip()],
        group_id=None,
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        auto_offset_reset="latest",
        enable_auto_commit=False
    )
    producer = KafkaProducer(
        bootstrap_servers=[s.strip() for s in KAFKA_BOOTSTRAP_SERVERS.split(",") if s.strip()],
        value_serializer=lambda v: json.dumps(v).encode("utf-8")
    )

    print(f"Subscribed to topic '{KAFKA_TOPIC}'. Waiting for windows...")

    prediction_queue = deque()

    for msg in consumer:
        window = msg.value
        print("\nReceived window:", " | ".join(f"{d['ts'].split()[-1]} {d['cpu_pct']:.2f}" for d in window))
        window_scaled = scale_window(window, scaler, SCALER_FEATURE_NAME)
        print("\nScaled window:", " | ".join(f"{x:.4f}" for x in window_scaled))
        inference_start = time.time()
        pred = send_inference(window_scaled)
        latency_ms = int((time.time() - inference_start) * 1000)
        # Compute the "prediction_for" timestamp by adding the horizon to last ts
        ts_last = pd.Timestamp(window[-1]['ts'])
        ts_pred_for = (ts_last + pd.Timedelta(seconds=FORECAST_HORIZON_SEC)).strftime("%Y-%m-%d %H:%M:%S")
        ts_pred_made_at = window[-1]['ts']

        if pred is not None:
            pred_rescaled = scaler.inverse_transform(
                pd.DataFrame([[pred]], columns=[SCALER_FEATURE_NAME])
            )[0, 0]
            prediction_queue.append({
                'prediction_for': ts_pred_for,
                'prediction_made_at': ts_pred_made_at,
                'forecasted_value': pred_rescaled,
                'latency': latency_ms
            })
            print(
                f"Prediction for timestamp {ts_pred_for}: {pred_rescaled:.2f} [cpu %] (latency: {latency_ms} ms)"
            )

        # Now check if we have a previous prediction to score (i.e., there are at least 2 items in queue)
        if len(prediction_queue) > 1:
            pending = prediction_queue.popleft()
            # Ground truth for the previous prediction is the *first* value of this window
            gt_ts = window[-1]["ts"]  # Actually, if your windows slide by one, GT is in the first element of current window
            for d in window:
                if d['ts'] == pending['prediction_for']:
                    gt_val = d['cpu_pct']
                    break
            else:
                # If not found, use last as fallback (shouldn't happen unless windows jump)
                gt_val = window[0]['cpu_pct']
            error = abs(pending['forecasted_value'] - gt_val)
            print(
                f"[DEBUG] Previous prediction was for {pending['prediction_for']} (made at {pending['prediction_made_at']}). "
                f"GT: {gt_val:.2f}, Error: {error:.4f}"
            )
            publish_forecast_and_metric(
                producer,
                pred_for=pending['prediction_for'],
                pred_made_at=pending['prediction_made_at'],
                forecasted_value=pending['forecasted_value'],
                real_value=gt_val,
                error=error,
                latency=pending['latency']
            )
        print("-----------------------------------------------------------------------------------------------\n")

if __name__ == "__main__":
    main()


Scaler loaded from MinIO.
Subscribed to topic 'sliding_window_lstm'. Waiting for windows...

Received window: 10:04:50 20.76 | 10:04:55 20.76 | 10:05:00 20.76 | 10:05:05 20.76 | 10:05:20 19.93

Scaled window: 0.2148 | 0.2148 | 0.2148 | 0.2148 | 0.2058
Prediction for timestamp 2025-10-27 10:05:25: 20.16 [cpu %] (latency: 397 ms)
-----------------------------------------------------------------------------------------------


Received window: 10:04:55 20.76 | 10:05:00 20.76 | 10:05:05 20.76 | 10:05:20 19.93 | 10:05:25 19.93

Scaled window: 0.2148 | 0.2148 | 0.2148 | 0.2058 | 0.2058
Prediction for timestamp 2025-10-27 10:05:30: 19.97 [cpu %] (latency: 11 ms)
[DEBUG] Previous prediction was for 2025-10-27 10:05:25 (made at 2025-10-27 10:05:20). GT: 19.93, Error: 0.2293
[DEBUG] Publishing forecast to topic: lstm_forecast_values
{'prediction_for': '2025-10-27 10:05:25', 'prediction_made_at': '2025-10-27 10:05:20', 'forecasted_value': 20.159983038902283, 'real_value': 19.9307291666734}
[DEBUG